# YOLOv8 Detection of Objects in video

Possible Courses of action to test
1. 2 Shot: 1) Detect all objects, then 2) Embedded all objects in individually
2. 1 Shot 1) Embed the entire image for similarity

Goal: To Detect when the cleaner's car has arrived at the cabin once detection has been detected.

In [1]:
from ultralytics import YOLO
import pandas as pd
from uuid import uuid4
from datetime import datetime
from torchvision import transforms
from PIL import Image
import torch
import cv2
import os
from concurrent.futures import ThreadPoolExecutor
from itertools import repeat
import numpy as np

In [2]:
# import my custom db functions
import database

Load the model to run on media

In [2]:
model_path = './models/yolov8m.pt'

In [3]:
# Load a REGULAR model
reg_model = YOLO(model_path, 'detect')  # pretrained YOLOv8n model
reg_model.to('cuda:0')

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_

In [4]:
# Load EMBEDDING model
emb_model = YOLO(model_path, 'detect')  # pretrained YOLOv8n model
emb_model.to('cuda:0')
emb_model.model.model = emb_model.model.model[:-1]

Set the location of the media

In [5]:
dir_path = "./cleaner_video-20230918/*.jpg"

In [ ]:
# clear database
# database.clear_and_delete_all_data()

Run the model with stream False to save images

200 images completes in 14.3 seconds

In [117]:
## Set default values
im_size = 640
test_img = '/mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0393.jpg'

#### Using PIL

In [27]:
pil_img = Image.open(test_img).resize((im_size, im_size))
tensor_img = transforms.functional.pil_to_tensor(pil_img)[None, :] / 255.0
tensor_img = tensor_img.to('cuda:0')

In [28]:
# run the model with the head removed
results = reg_model.model(tensor_img)
# Reshape the tensor to separate the 576 matrices
reshaped_tensor = results.view(1, 576, -1)  # Reshape to [1, 576, 400]
print(reshaped_tensor.shape)

# Calculate the mean along the last dimension (the 20x20 matrices)
mean_values = torch.mean(reshaped_tensor, dim=2)
print(mean_values.shape)

torch.Size([1, 576, 400])
torch.Size([1, 576])


#### Using OpenCV

In [73]:
# Load the image using OpenCV
opencv_img = cv2.imread(test_img)
# Resize the image
opencv_img = cv2.resize(opencv_img, (im_size, im_size))
# Convert from BGR to RGB (OpenCV loads images in BGR format by default)
opencv_img = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2RGB)

# Convert the image to a PyTorch tensor and normalize it to [0, 1]
tensor_img = torch.tensor(opencv_img, dtype=torch.float32).permute(2, 0, 1) / 255.0
tensor_img = tensor_img.unsqueeze(0)  # Add a batch dimension

# Move the tensor to GPU if needed
tensor_img = tensor_img.to('cuda:0')

In [74]:
# run the model with the head removed
results = emb_model.model(tensor_img)
# Reshape the tensor to separate the 576 matrices
reshaped_tensor = results.view(1, 576, -1)  # Reshape to [1, 576, 400]
print(reshaped_tensor.shape)

# Calculate the mean along the last dimension (the 20x20 matrices)
mean_values = torch.mean(reshaped_tensor, dim=2)
print(mean_values.shape)

torch.Size([1, 576, 400])
torch.Size([1, 576])


### Batch the Images into tensors for the Embedding

In [6]:
def preprocess_image_single(input_image, im_size=640):
    """
    Preprocess a single image using OpenCV, ensuring that it matches the desired size (640x640) with padding.
    Accepts either a file path or an already opened OpenCV image.

    Args:
        input_image (str or numpy.ndarray): Either a file path to the image or an already opened OpenCV image.
        im_size (int): Desired image size (both width and height).

    Returns:
        Preprocessed image tensor.
    """
    if isinstance(input_image, str):
        # Load the image using OpenCV if input is a file path
        if not os.path.isfile(input_image):
            raise FileNotFoundError(f"Image file not found at: {input_image}")
        opencv_img = cv2.imread(input_image)
    elif isinstance(input_image, np.ndarray):
        # Use the provided OpenCV image if input is already opened
        opencv_img = input_image
    else:
        raise ValueError("Input must be a file path (str) or an already opened OpenCV image (numpy.ndarray).")

    # Calculate the new dimensions while preserving the original aspect ratio
    height, width, _ = opencv_img.shape
    aspect_ratio = width / height
    
    if aspect_ratio > 1:
        # Landscape orientation
        new_width = im_size
        new_height = int(new_width / aspect_ratio)
    else:
        # Portrait orientation
        new_height = im_size
        new_width = int(new_height * aspect_ratio)
    
    # Resize the image
    opencv_img = cv2.resize(opencv_img, (new_width, new_height))
    
    # Create a blank canvas of the desired size and paste the resized image onto it
    padded_img = torch.zeros(3, im_size, im_size, dtype=torch.float32)
    start_h = (im_size - new_height) // 2
    start_w = (im_size - new_width) // 2
    padded_img[:, start_h:start_h+new_height, start_w:start_w+new_width] = torch.tensor(opencv_img, dtype=torch.float32).permute(2, 0, 1) / 255.0
    
    return padded_img


In [12]:
def load_and_preprocess_images(input_paths, im_size=640, device='cuda:0'):
    """
    Load and preprocess images from a list of image paths or a directory path using parallel processing,
    and return a batch tensor in BCHW format with padding to the right and bottom.

    Args:
        input_paths (str or list): A single image path as a string, a list of image paths, or a directory path.
        im_size (int): Desired image size (both width and height).
        device (str): Device to move the tensors to (e.g., 'cuda:0' for GPU).
        num_processes (int): Number of parallel processes to use for image loading.

    Returns:
        Batch tensor in BCHW format containing the preprocessed images with padding.
    """
    if isinstance(input_paths, str):
        if os.path.isfile(input_paths):
            # Single image file
            image_paths = [input_paths]
        elif os.path.isdir(input_paths):
            # Directory containing image files
            image_paths = [os.path.join(input_paths, file) for file in os.listdir(input_paths) if file.endswith('.jpg')]
        else:
            raise ValueError("Input path must be a valid file or directory.")
    elif isinstance(input_paths, list):
        image_paths = input_paths
    elif isinstance(input_paths, np.ndarray):
        # Use the provided OpenCV image if input is already opened
        image_paths = [input_paths]
    else:
        raise ValueError("Input paths must be a string, a list of strings, or a directory path.")

    # Create a thread pool to parallelize image loading and preprocessing      
    with ThreadPoolExecutor() as executor:
        images_resized = executor.map(
            preprocess_image_single, image_paths, repeat(im_size)
        )

    tensor_images = list(images_resized)

    # Stack the individual tensors to create a batch tensor
    batch_tensor = torch.stack(tensor_images, dim=0)
    
    # Move the batch tensor to the specified device
    batch_tensor = batch_tensor.to(device)

    return image_paths, batch_tensor


In [13]:
def process_tensor_and_calculate_mean(model, input_tensor):
    """
    Process an input tensor through a model with the head removed, reshape the output,
    and calculate the mean along the last dimension of the reshaped tensor for each sample in the batch.

    Args:
        model (nn.Module): The PyTorch model with the head removed.
        input_tensor (torch.Tensor): Input tensor to be processed.

    Returns:
        torch.Tensor: A tensor containing the mean values along the last dimension for each sample in the batch.
    """
    # Ensure the input tensor has the correct shape (BCHW)
    if len(input_tensor.shape) != 4:
        raise ValueError("Input tensor should have shape (batch_size, channels, height, width).")

    # Run the input tensor through the model with the head removed
    results = model.model(input_tensor)
    
    # Reshape the tensor to separate the 576 matrices
    batch_size, num_matrices, matrix_size, _ = results.shape
    reshaped_tensor = results.view(batch_size, num_matrices, -1)  # Reshape to [batch_size, 576, 400]
    
    # Calculate the mean along the last dimension (the 20x20 matrices) for each sample in the batch
    mean_values = torch.mean(reshaped_tensor, dim=2)
    
    # Normalize each vector by dividing it by its L2 norm (Euclidean norm)
    normed_mean_values = mean_values / torch.norm(mean_values, p=2, dim=1, keepdim=True)
    
    return normed_mean_values

In [14]:
def get_embeddings(image_paths, emb_model, im_size=640, device='cuda:0'):
    """
    Load and preprocess a batch of images, then calculate feature embeddings for the batch using a model.

    Args:
        image_paths (list): A list of image paths.
        emb_model (nn.Module): The PyTorch model used for feature embedding.
        im_size (int): Desired image size (both width and height).
        device (str): Device to move the tensors to (e.g., 'cuda:0' for GPU).

    Returns:
        torch.Tensor: Batch of feature embeddings.
    """
    # Load and preprocess the images
    bath_paths, batch_tensor = load_and_preprocess_images(image_paths, im_size=im_size, device=device)
    
    # Calculate feature embeddings for the batch
    batch_feature_embeddings = process_tensor_and_calculate_mean(emb_model, batch_tensor)
    
    return bath_paths, batch_feature_embeddings

In [140]:
# List of test images for testing
test_image_paths = [ele for ele in [test_img] for i in range(10)]

In [25]:
img_paths, img_embeddings = get_embeddings("./training_vehicle/", emb_model)

In [33]:
img_embeddings.shape

torch.Size([206, 576])

In [36]:
emb_list = img_embeddings.tolist()

In [37]:
len(emb_list)

206

In [40]:
ref_vec = torch.mean(img_embeddings, 0).tolist()

In [42]:
str(ref_vec)

'[0.026859046891331673, 0.021249866113066673, 0.0037175556644797325, -0.026432767510414124, 0.06306134164333344, -0.01879010908305645, 0.013398168608546257, 0.05031919106841087, 0.03170360252261162, -0.04144864156842232, 0.006283950991928577, -0.017785727977752686, 0.040287137031555176, 0.047069355845451355, 0.0331176221370697, -0.017067013308405876, 0.046757813543081284, -0.052207961678504944, -0.03392577916383743, -0.032655004411935806, 0.022627925500273705, -0.003949647303670645, -0.013088181614875793, -0.030059387907385826, -0.0005968641489744186, -0.0024588739033788443, 0.043740179389715195, -0.006358692888170481, 0.007605380844324827, -0.02348916605114937, -0.035635676234960556, -0.014236696064472198, -0.06251750141382217, -0.02424663119018078, 0.08646659553050995, 0.05970676243305206, -0.03303743898868561, 0.0007757276180200279, 0.009273829869925976, 0.016681265085935593, 0.053435068577528, -0.05822105333209038, -0.014273208566009998, 0.08686665445566177, 0.023173058405518532, -

In [156]:
for vec in img_embeddings:
    print(vec)

tensor([ 3.1553e-02,  3.7047e-02,  2.4202e-02, -1.8097e-02, -2.2204e-02, -4.1388e-02,  5.2603e-02,  4.5294e-02,  5.0175e-02, -3.6117e-02,  3.1640e-02, -3.5843e-02,  5.4900e-02,  8.3680e-02,  6.1769e-02, -1.2868e-02,  1.8877e-02, -4.3514e-02, -9.5013e-03, -5.0842e-02,  5.1173e-02,  2.9154e-02, -1.1316e-02,  4.7410e-03,
        -1.6104e-03,  2.3071e-02,  6.6219e-02, -3.4131e-02,  7.7498e-02, -3.9390e-02, -5.0763e-02, -3.3638e-03, -5.9075e-02, -2.0850e-02,  3.7009e-02,  5.2005e-02, -3.0666e-02,  1.8775e-02,  2.8771e-02,  4.3222e-02,  4.6001e-02, -3.5228e-02,  2.1455e-03,  6.1029e-02,  3.1845e-02, -4.0542e-02,  3.9502e-02, -1.8017e-03,
         8.8261e-03,  3.1304e-02,  2.4753e-03, -4.1785e-02,  4.6136e-02,  2.6410e-02,  6.2116e-02,  4.5037e-02,  5.5241e-02,  3.1579e-02,  4.7166e-02, -1.5511e-02,  8.0168e-02,  7.2420e-02,  1.3327e-02, -2.9883e-02,  3.6641e-02,  8.5437e-02,  2.7279e-02,  3.8040e-02,  4.2828e-02,  7.3123e-02,  8.9507e-03, -6.7765e-03,
         3.1129e-02,  9.7811e-02,  8.706

### Using regular predict on YOLO Model

In [33]:
results = reg_model.predict(test_img, imgsz = 640, save_crop = True, classes = list(range(9)), stream=False) 

IndexError: The shape of the mask [14, 20] at index 0 does not match the shape of the indexed tensor [576, 20, 14] at index 0

In [8]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

In [ ]:
# Run batched inference on a list of images
results = model.predict(dir_path, imgsz = 640, save_crop = True, classes = list(range(9)), stream=False)  # return a generator of Results objects

Example output loop for getting data out of prediction results

In [ ]:
# Process results generator
for result in results:
    boxes = result.boxes  # Boxes object for bbox outputs
    probs = result.probs  # Probs object for classification outputs


## Loop through the Results to a dataframe

In [ ]:
df_list = []
df_metadata = []
index = 0
for m_index, result in enumerate(results):
    file_uuid = uuid4().hex
    m_df = pd.DataFrame({'file_uuid': file_uuid, 'date': datetime.now(), 'path': result.path, 'height': result.orig_shape[0], 'width': result.orig_shape[1]}, index=[m_index])
    boxes = result.boxes
    for box in boxes:
        cls = int(box.cls[0])
        class_name = model.names[cls]
        conf = int(box.conf[0]*100)
        bx = box.xywhn.tolist()[0]
        gen_uuid = uuid4().hex
        df = pd.DataFrame({'uuid': gen_uuid, 'file_uuid': file_uuid, 'class_name': class_name, 'class_id': cls, 'confidence': conf, 'xcenter': bx[0], 'ycenter': bx[1], 'width': bx[2], 'height': bx[3]}, index = [index])
        df_list.append(df)
        index += 1
    df_metadata.append(m_df)

# create dataframes
df_meta = pd.concat(df_metadata)
df = pd.concat(df_list)

In [ ]:
# database the metadata
database.insert_metadata_bulk(df_meta)

In [ ]:
# database the detections
database.insert_detections_bulk(df)

In [ ]:
df